In [ ]:
!pip install -q torch numpy scipy pillow matplotlib opencv-python huggingface_hub \
    safetensors einops "git+https://github.com/facebookresearch/vggt"

In [ ]:
import tarfile
import time
import urllib.request
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from scipy.spatial.transform import Rotation
from vggt.models.vggt import VGGT
from vggt.utils.geometry import unproject_depth_map_to_point_map
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
MODEL_NAME = "facebook/VGGT-1B"
DATA_DIR = Path("../data/tum")
SEQUENCE = "rgbd_dataset_freiburg1_desk"
SEQUENCE_URL = f"https://cvg.cit.tum.de/rgbd/dataset/freiburg1/{SEQUENCE}.tgz"
# Kinect depth PNGs store millimetres x 5
DEPTH_SCALE = 5000.0
# Factory calibration of the freiburg1 Kinect at 640 x 480
TUM_K = np.array([[517.3, 0.0, 318.6], [0.0, 516.5, 255.3], [0.0, 0.0, 1.0]])
NUM_VIEWS = 8
# Frames 0..160 of the sequence: about 5 s of hand-held motion over a desk
FRAME_SPAN = 160
SEED = 0

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"DEVICE = {DEVICE}")

# 3D Vision: Feed-Forward Reconstruction

Recovering cameras and geometry from a handful of photos used to mean a long optimisation per scene.
This notebook rebuilds the geometry around **VGGT**, a single transformer that predicts every
camera, depth map and 3D point map for a set of unposed images in one forward pass. It then
measures how accurate that pass is against motion-capture poses and Kinect depth.

---

# 1) Introduction

## 1.1 The problem

Given $S$ images of a static scene with **unknown** cameras, recover

- each camera's pose $[R_i \mid t_i]$ and intrinsics $K_i$,
- a dense depth map $D_i$ per image,
- one shared 3D point cloud.

## 1.2 Why the classical pipeline falls short

| Stage | Classical (COLMAP-style SfM + MVS) | What goes wrong |
|---|---|---|
| Correspondences | Hand-crafted keypoints (SIFT/ORB) + matching | Few matches on texture-less walls, repeated patterns, wide baselines |
| Cameras | Incremental registration + bundle adjustment | Minutes per scene; brittle initialisation; fails with 2–5 views |
| Dense geometry | Multi-view stereo or per-scene NeRF / 3DGS optimisation | Minutes to hours; needs the poses from the stage before |

Every stage is a separate optimisation, so errors early on cannot be corrected later.

## 1.3 The core idea

Learn the whole mapping instead:

$$
f_\theta:\ \{I_i\}_{i=1}^{S} \;\longmapsto\; \{\,g_i,\ D_i,\ \Sigma^D_i,\ P_i,\ \Sigma^P_i\,\}_{i=1}^{S}
$$

where $g_i$ encodes camera $i$, $D_i$ is its depth map, $P_i \in \mathbb{R}^{H\times W\times 3}$ is a
**point map** giving each pixel's 3D position in the first camera's frame, and $\Sigma$ are
per-pixel confidences. DUSt3R (2024) did this for image **pairs**. VGGT (CVPR 2025 best paper)
does it for 1 to hundreds of views with one transformer that alternates **frame-wise** and
**global** attention.

```
images (S, 3, H, W)
   │  DINOv2 patchify (14 px) + 1 camera token + 4 register tokens per frame
   ▼
tokens (S, 5 + H/14·W/14, 1024)
   │  24 × [ frame attention (within one image) → global attention (across all images) ]
   ▼
camera head ── g_i = [t (3), quaternion (4), FoV_h, FoV_w]      (S, 9)
depth head  ── D_i, Σ^D_i   (DPT decoder)                        (S, H, W)
point head  ── P_i, Σ^P_i   (DPT decoder)                        (S, H, W, 3)
```

The first frame gets its own learned camera token, which is how the network knows which camera
defines the world frame.

## 1.4 Findings from the past year

| Model / method | Date | What is new |
|---|---|---|
| **VGGT** (Meta / Oxford) | 2025 | Feed-forward transformer predicting cameras, depth maps, point maps and point tracks from 1 to hundreds of views; also a strong feature backbone for novel-view synthesis |
| **VGGS** | AAAI 2026 | Gaussian splatting guided by VGGT priors for sparse-view surface reconstruction; 4–7× faster optimisation with lower memory |
| Feed-forward Gaussian splatting | 2025–26 | Predict Gaussian parameters directly from unposed images (e.g. VGGT-based branches with photometric supervision) |
| **DUSt3R / MASt3R** | 2024 | Pairwise point-map regression; the paradigm VGGT generalises |
| Survey | 2025 | Feed-forward 3D reconstruction grouped by NeRF / point-map / 3DGS representations (arXiv 2507.14501) |

## 1.5 What this notebook covers

| Section | Content |
|---|---|
| 2) Setup | TUM RGB-D `fr1/desk` (real Kinect depth + motion-capture poses), `facebook/VGGT-1B` |
| 3) The math | Unprojection, point maps, camera encoding, confidence-weighted loss, Umeyama alignment, metrics |
| 4) From scratch | Pose decoding, unprojection, Umeyama, metrics, confidence loss, each checked against a reference |
| 5) End-to-end | One VGGT pass on 8 views: cameras, depth, point clouds, confidence vs error |
| 6) Variants | Classical ORB + essential-matrix baseline; accuracy, time and memory vs number of views |
| 7) Practical notes | Memory, scale ambiguity, failure modes, Gaussian-splatting hand-off |
| 8) Conclusion | Decision table |

---

# 2) Setup

## 2.1 Data: TUM RGB-D `freiburg1_desk`

A hand-held Kinect sweeps over an office desk. Each RGB frame has a depth map from the Kinect's
structured-light sensor and a 6-DoF pose from an external motion-capture system. Those are the two
ground truths we score against. The archive is ~360 MB. Streams run at different rates, so frames
are paired by nearest timestamp.

In [ ]:
def download_sequence() -> Path:
    """Download and extract the TUM sequence once; return its folder."""
    root = DATA_DIR / SEQUENCE
    if not root.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        archive = DATA_DIR / f"{SEQUENCE}.tgz"
        urllib.request.urlretrieve(SEQUENCE_URL, archive)
        with tarfile.open(archive) as tar:
            tar.extractall(DATA_DIR)
        archive.unlink()
    return root


def read_index(path: Path) -> list[list[str]]:
    """Rows of a TUM text index, skipping '#' comment lines."""
    lines = path.read_text().splitlines()
    return [line.split() for line in lines if not line.startswith("#")]


def associate_frames(root: Path, max_dt: float = 0.02) -> list[dict]:
    """Pair each RGB frame with the nearest depth map and mocap pose (gap <= max_dt)."""
    depth_rows = read_index(root / "depth.txt")
    depth_times = np.array([float(row[0]) for row in depth_rows])
    poses = np.array(
        [[float(v) for v in row] for row in read_index(root / "groundtruth.txt")]
    )
    frames = []
    for stamp, rgb_file in read_index(root / "rgb.txt"):
        t = float(stamp)
        j = np.abs(depth_times - t).argmin()
        k = np.abs(poses[:, 0] - t).argmin()
        if abs(depth_times[j] - t) < max_dt and abs(poses[k, 0] - t) < max_dt:
            frames.append(
                {
                    "rgb": root / rgb_file,
                    "depth": root / depth_rows[j][1],
                    "pose": poses[k, 1:],
                }
            )
    return frames


root = download_sequence()
frames = associate_frames(root)
frame_idx = np.linspace(0, FRAME_SPAN, NUM_VIEWS).round().astype(int)
views = [frames[i] for i in frame_idx]
print(f"associated frames: {len(frames)}")
print(f"selected indices : {frame_idx.tolist()}")
print(f"first pose [tx ty tz qx qy qz qw]: {np.round(views[0]['pose'], 4)}")

In [ ]:
fig, axes = plt.subplots(2, NUM_VIEWS // 2, figsize=(14, 5))
for ax, view, idx in zip(axes.flat, views, frame_idx):
    ax.imshow(Image.open(view["rgb"]))
    ax.set_title(f"frame {idx}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2.2 Ground truth in the model's conventions

VGGT's world frame is **the first camera**, in OpenCV axes (x right, y down, z forward). Its
extrinsics map world to camera. TUM poses are camera-to-mocap-world. Re-expressing every
ground-truth pose relative to the first frame puts both in the same frame:

$$
T^{\text{gt}}_{i \to 1} = \big(T^{\text{gt}}_{1 \to w}\big)^{-1} T^{\text{gt}}_{i \to w}
$$

VGGT resizes 640 × 480 frames to width 518 with height rounded to a multiple of the 14-pixel patch
(392). The ground-truth depth and intrinsics are resized to the same grid.

In [ ]:
def pose_to_c2w(pose: np.ndarray) -> np.ndarray:
    """(7,) TUM pose [tx, ty, tz, qx, qy, qz, qw] -> (4, 4) camera-to-world matrix."""
    c2w = np.eye(4)
    c2w[:3, :3] = Rotation.from_quat(pose[3:]).as_matrix()
    c2w[:3, 3] = pose[:3]
    return c2w


def load_gt_depth(path: Path, height: int, width: int) -> np.ndarray:
    """Kinect PNG -> (height, width) depth in metres; 0 marks missing depth."""
    depth = np.asarray(Image.open(path), dtype=np.float32) / DEPTH_SCALE
    # nearest keeps holes as holes instead of blending them into neighbours
    depth = torch.from_numpy(depth)[None, None]
    return F.interpolate(depth, size=(height, width), mode="nearest")[0, 0].numpy()


images = load_and_preprocess_images([str(view["rgb"]) for view in views])
height, width = images.shape[-2:]

# (num_views, 4, 4) camera-to-first-camera transforms
gt_c2w = np.stack([pose_to_c2w(view["pose"]) for view in views])
gt_c2w = np.linalg.inv(gt_c2w[0]) @ gt_c2w
gt_depth = np.stack([load_gt_depth(view["depth"], height, width) for view in views])
gt_K = TUM_K.copy()
gt_K[0] *= width / 640
gt_K[1] *= height / 480

print(f"images   : {tuple(images.shape)}  (num_views, 3, height, width), range [0, 1]")
print(
    f"gt depth : {gt_depth.shape}, valid {np.mean(gt_depth > 0):.1%}, "
    f"median {np.median(gt_depth[gt_depth > 0]):.2f} m"
)
print(
    "gt path length: "
    f"{np.linalg.norm(np.diff(gt_c2w[:, :3, 3], axis=0), axis=1).sum():.2f} m"
)
print(f"gt K at model resolution:\n{np.round(gt_K, 1)}")

## 2.3 Model

`facebook/VGGT-1B` (~1.26 B parameters, 5 GB in fp32). The aggregator holds almost all of the
weights and runs in fp16. The heads stay in fp32, matching the reference implementation, which
disables autocast for the heads so the `exp` activations that produce depth and confidence are
computed in full precision.

In [ ]:
def device_memory_gb() -> float:
    """Memory currently held by the accelerator, in GB."""
    if DEVICE.type == "cuda":
        return torch.cuda.max_memory_allocated() / 1e9
    if DEVICE.type == "mps":
        return torch.mps.driver_allocated_memory() / 1e9
    return 0.0


start = time.time()
model = VGGT.from_pretrained(MODEL_NAME).eval()
model.aggregator.half()
model.to(DEVICE)

for name in ["aggregator", "camera_head", "depth_head", "point_head", "track_head"]:
    num_params = sum(p.numel() for p in getattr(model, name).parameters())
    print(f"{name:12s} {num_params / 1e6:7.1f} M params")
print(
    f"load time {time.time() - start:.1f} s | device memory {device_memory_gb():.2f} GB"
)

---

# 3) The math

## 3.1 Pinhole unprojection

A pixel $(u, v)$ with depth $D(u,v)$ in a camera with intrinsics
$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$ sits at

$$
\mathbf{x}^{\text{cam}} = D(u,v)\, K^{-1} \begin{bmatrix} u \\ v \\ 1 \end{bmatrix}
= D \begin{bmatrix} (u - c_x)/f_x \\ (v - c_y)/f_y \\ 1 \end{bmatrix}.
$$

With extrinsics $[R \mid t]$ mapping world to camera ($\mathbf{x}^{\text{cam}} = R\mathbf{x}^{\text{w}} + t$),
the world point is $\mathbf{x}^{\text{w}} = R^\top(\mathbf{x}^{\text{cam}} - t)$.

## 3.2 Two routes to the same point cloud

VGGT predicts geometry twice:

| Route | Output | How a 3D point is obtained |
|---|---|---|
| Point head | $P_i(u,v) \in \mathbb{R}^3$ directly in frame 1 | read it off |
| Depth + camera heads | $D_i(u,v)$ and $g_i$ | unproject with §3.1 |

The two are redundant at test time but not during training: predicting both gives extra
supervision that improves the shared trunk. The paper reports that the depth + camera route is
the more accurate of the two, which §5.4 checks.

## 3.3 Camera encoding

Each camera is regressed as a 9-vector

$$
g_i = [\,t_i \in \mathbb{R}^3,\ q_i \in \mathbb{R}^4,\ \phi^h_i,\ \phi^w_i\,],
$$

with translation $t_i$, a quaternion $q_i = (x, y, z, w)$ (scalar last) for $R_i$, and vertical and
horizontal fields of view. The principal point is assumed to be the image centre, and

$$
f_y = \frac{H/2}{\tan(\phi^h/2)}, \qquad f_x = \frac{W/2}{\tan(\phi^w/2)}.
$$

FoV is used instead of focal length because it does not depend on image resolution. The first
camera is the identity by construction: $t_1 = 0,\ q_1 = (0,0,0,1)$.

The unit quaternion $q = (x, y, z, w)$ gives

$$
R(q) = \begin{bmatrix}
1 - 2(y^2+z^2) & 2(xy - zw) & 2(xz + yw)\\
2(xy + zw) & 1 - 2(x^2+z^2) & 2(yz - xw)\\
2(xz - yw) & 2(yz + xw) & 1 - 2(x^2+y^2)
\end{bmatrix}.
$$

## 3.4 Confidence-weighted regression (the training loss)

Depth and point maps are trained with an aleatoric-uncertainty loss. For a predicted point
$\hat{\mathbf{x}}_p$, target $\mathbf{x}_p$ and confidence $c_p > 0$ at pixel $p$:

$$
\mathcal{L}_{\text{conf}} = \sum_p \Big( c_p\, \lVert \hat{\mathbf{x}}_p - \mathbf{x}_p \rVert
+ c_p\, \lVert \nabla\hat{\mathbf{x}}_p - \nabla\mathbf{x}_p \rVert \;-\; \alpha \log c_p \Big).
$$

The second term matches image gradients of the map, which keeps surfaces sharp. Minimising over
$c_p$ with the error $e_p$ fixed gives

$$
\frac{\partial}{\partial c_p}\big(c_p e_p - \alpha \log c_p\big) = 0 \;\Rightarrow\; c_p^\star = \frac{\alpha}{e_p}.
$$

So the network learns to output confidence **inversely proportional to its expected error**.
That is a testable claim, and §5.5 tests it on real data. Ground truth is normalised by the
mean distance of all points to the origin, so predictions are only defined **up to a global scale**.

## 3.5 Evaluation needs a similarity alignment (Umeyama)

Because scale and world frame are arbitrary, predicted points $\{\mathbf{x}_i\}$ are aligned to
ground truth $\{\mathbf{y}_i\}$ by the best similarity transform before measuring error:

$$
(s^\star, R^\star, t^\star) = \arg\min_{s, R, t} \frac{1}{n}\sum_{i=1}^{n} \lVert \mathbf{y}_i - (sR\mathbf{x}_i + t)\rVert^2 .
$$

With means $\mu_x, \mu_y$, variance $\sigma_x^2 = \frac{1}{n}\sum\lVert\mathbf{x}_i-\mu_x\rVert^2$
and cross-covariance $\Sigma_{xy} = \frac{1}{n}\sum (\mathbf{y}_i-\mu_y)(\mathbf{x}_i-\mu_x)^\top = UDV^\top$
(SVD), the closed form is

$$
S = \operatorname{diag}\big(1, 1, \det(UV^\top)\big),\quad
R^\star = USV^\top,\quad
s^\star = \frac{\operatorname{tr}(DS)}{\sigma_x^2},\quad
t^\star = \mu_y - s^\star R^\star \mu_x .
$$

$S$ flips the last axis when the SVD would otherwise return a reflection.

## 3.6 Metrics

| Metric | Definition | Measures |
|---|---|---|
| ATE | $\sqrt{\frac{1}{S}\sum_i \lVert s R \hat{c}_i + t - c_i\rVert^2}$ over camera centres $c_i$ after Umeyama | global trajectory drift (m) |
| RRE | $\angle\big(\hat{R}_{ij}^\top R_{ij}\big)$ for every pair $i<j$, $R_{ij}=R_i^\top R_j$ | relative rotation error (°), frame-free |
| AbsRel | $\frac{1}{N}\sum_p \lvert s\hat{D}_p - D_p\rvert / D_p$, $s$ = median ratio | per-image depth accuracy |
| $\delta_1$ | fraction of pixels with $\max(s\hat{D}_p/D_p,\ D_p/s\hat{D}_p) < 1.25$ | depth inlier rate |
| Chamfer | $\frac{1}{2}\big(\operatorname{mean}_{x\in A}\min_{y\in B}\lVert x-y\rVert + \operatorname{mean}_{y\in B}\min_{x\in A}\lVert x-y\rVert\big)$ | point-cloud agreement (m) |

---

# 4) From-scratch implementation

## 4.1 Decoding the camera encoding

The decoder turns the 9-vector into a world-to-camera matrix $[R \mid t]$ and a pinhole $K$
(§3.3). It is checked against `vggt.utils.pose_enc.pose_encoding_to_extri_intri` on a real
encoding: the ground-truth cameras of our 8 views, encoded the way VGGT encodes training targets.

In [ ]:
def quaternion_to_matrix(quat: torch.Tensor) -> torch.Tensor:
    """(..., 4) quaternion (x, y, z, w), any norm -> (..., 3, 3) rotation matrix."""
    quat = quat / quat.norm(dim=-1, keepdim=True)
    x, y, z, w = quat.unbind(-1)
    rows = [
        1 - 2 * (y * y + z * z),
        2 * (x * y - z * w),
        2 * (x * z + y * w),
        2 * (x * y + z * w),
        1 - 2 * (x * x + z * z),
        2 * (y * z - x * w),
        2 * (x * z - y * w),
        2 * (y * z + x * w),
        1 - 2 * (x * x + y * y),
    ]
    return torch.stack(rows, dim=-1).reshape(*quat.shape[:-1], 3, 3)


def decode_camera(
    pose_enc: torch.Tensor, height: int, width: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """(num_views, 9) [t, quat, fov_h, fov_w] -> extrinsics, K.

    Extrinsics are (num_views, 3, 4) world-to-camera, K is (num_views, 3, 3).
    """
    trans, quat, fov_h, fov_w = (
        pose_enc[:, :3],
        pose_enc[:, 3:7],
        pose_enc[:, 7],
        pose_enc[:, 8],
    )
    extrinsics = torch.cat([quaternion_to_matrix(quat), trans[:, :, None]], dim=-1)
    intrinsics = torch.zeros(len(pose_enc), 3, 3, dtype=pose_enc.dtype)
    intrinsics[:, 0, 0] = (width / 2) / torch.tan(fov_w / 2)
    intrinsics[:, 1, 1] = (height / 2) / torch.tan(fov_h / 2)
    intrinsics[:, 0, 2] = width / 2
    intrinsics[:, 1, 2] = height / 2
    intrinsics[:, 2, 2] = 1.0
    return extrinsics, intrinsics


def encode_camera(
    w2c: np.ndarray, K: np.ndarray, height: int, width: int
) -> torch.Tensor:
    """(num_views, 3, 4) world-to-camera + (3, 3) K -> (num_views, 9) encoding.

    Same layout as VGGT's training targets.
    """
    quat = Rotation.from_matrix(w2c[:, :3, :3]).as_quat()
    fov_h = 2 * np.arctan((height / 2) / K[1, 1])
    fov_w = 2 * np.arctan((width / 2) / K[0, 0])
    fovs = np.tile([fov_h, fov_w], (len(w2c), 1))
    return torch.tensor(
        np.concatenate([w2c[:, :3, 3], quat, fovs], axis=1), dtype=torch.float32
    )

Round trip: encode the ground-truth cameras, decode them with our function and with VGGT's.

In [ ]:
gt_w2c = np.linalg.inv(gt_c2w)[:, :3]
gt_enc = encode_camera(gt_w2c, gt_K, height, width)
ours_ext, ours_K = decode_camera(gt_enc, height, width)
ref_ext, ref_K = pose_encoding_to_extri_intri(gt_enc[None], (height, width))
print(f"encoding of view 1 (identity): {np.round(gt_enc[0].numpy(), 4)}")
print(f"max |extrinsics - reference| : {(ours_ext - ref_ext[0]).abs().max():.2e}")
print(f"max |K - reference|          : {(ours_K - ref_K[0]).abs().max():.2e}")
print(f"max |extrinsics - gt|        : {np.abs(ours_ext.numpy() - gt_w2c).max():.2e}")

## 4.2 Unprojecting depth into the shared frame

Vectorised over all views: build the pixel grid once, scale rays by depth, then apply
$R^\top(\mathbf{x}^{\text{cam}} - t)$. The ground-truth Kinect depth and mocap poses give real
input, and the result is compared with `vggt.utils.geometry.unproject_depth_map_to_point_map`.

In [ ]:
def unproject_depth(
    depth: torch.Tensor, extrinsics: torch.Tensor, intrinsics: torch.Tensor
) -> torch.Tensor:
    """depth (num_views, H, W), w2c (num_views, 3, 4), K (num_views, 3, 3).

    Returns world points (num_views, H, W, 3).
    """
    num_views, height, width = depth.shape
    v, u = torch.meshgrid(
        torch.arange(height, dtype=depth.dtype),
        torch.arange(width, dtype=depth.dtype),
        indexing="ij",
    )
    fx, fy = intrinsics[:, 0, 0, None, None], intrinsics[:, 1, 1, None, None]
    cx, cy = intrinsics[:, 0, 2, None, None], intrinsics[:, 1, 2, None, None]
    # (num_views, H, W, 3) camera-frame points
    cam = torch.stack([(u - cx) / fx * depth, (v - cy) / fy * depth, depth], dim=-1)
    rot, trans = extrinsics[:, :, :3], extrinsics[:, :, 3]
    # x_world = R^T (x_cam - t), batched over views and pixels
    return torch.einsum("sji,shwj->shwi", rot, cam - trans[:, None, None, :])


gt_points = unproject_depth(
    torch.from_numpy(gt_depth).double(),
    torch.from_numpy(gt_w2c),
    torch.from_numpy(np.tile(gt_K, (NUM_VIEWS, 1, 1))),
)
# the reference expects depth as (num_views, H, W, 1)
ref_points = unproject_depth_map_to_point_map(
    gt_depth[..., None], gt_w2c, np.tile(gt_K, (NUM_VIEWS, 1, 1))
)
gt_valid = gt_depth > 0
print(f"points: {tuple(gt_points.shape)}  (num_views, H, W, 3)")
print(
    f"max |ours - reference| on valid pixels: "
    f"{np.abs(gt_points.numpy() - ref_points)[gt_valid].max():.2e} m"
)

## 4.3 Umeyama similarity alignment

Test: apply a known similarity (scale 0.37, a random rotation, an offset) to the real ground-truth
camera centres, then recover it.

In [ ]:
def umeyama(src: np.ndarray, dst: np.ndarray) -> tuple[float, np.ndarray, np.ndarray]:
    """Best s, R, t with dst ≈ s R src + t; src, dst (n, 3) -> (s, (3, 3), (3,))."""
    mu_src, mu_dst = src.mean(0), dst.mean(0)
    src_c, dst_c = src - mu_src, dst - mu_dst
    cov = dst_c.T @ src_c / len(src)
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3)
    if np.linalg.det(U @ Vt) < 0:
        S[2, 2] = -1.0
    rot = U @ S @ Vt
    scale = np.trace(np.diag(D) @ S) / (src_c**2).sum(1).mean()
    trans = mu_dst - scale * rot @ mu_src
    return scale, rot, trans


gt_centres = gt_c2w[:, :3, 3]
true_rot = Rotation.random(random_state=SEED).as_matrix()
moved = 0.37 * gt_centres @ true_rot.T + np.array([1.0, -2.0, 0.5])
scale, rot, trans = umeyama(moved, gt_centres)
recovered = scale * moved @ rot.T + trans
print(f"recovered scale {scale:.4f} (expected {1 / 0.37:.4f})")
print(f"max residual {np.abs(recovered - gt_centres).max():.2e} m")

## 4.4 Metrics

ATE uses the camera centres $c_i = -R_i^\top t_i$. Chamfer distance on clouds of tens of thousands
of points is computed in chunks to bound the $n \times m$ distance matrix. Shifting a cloud by
1 cm gives a Chamfer distance just under 1 cm, because a point's nearest neighbour in the shifted
cloud is often closer than its own shifted copy.

In [ ]:
def camera_centres(extrinsics: np.ndarray) -> np.ndarray:
    """(num_views, 3, 4) world-to-camera -> (num_views, 3) centres c = -R^T t."""
    return -np.einsum("sji,sj->si", extrinsics[:, :, :3], extrinsics[:, :, 3])


def absolute_trajectory_error(
    pred_centres: np.ndarray, gt_centres: np.ndarray
) -> float:
    """RMSE of camera centres (m) after Umeyama alignment."""
    scale, rot, trans = umeyama(pred_centres, gt_centres)
    aligned = scale * pred_centres @ rot.T + trans
    return float(np.sqrt(((aligned - gt_centres) ** 2).sum(1).mean()))


def relative_rotation_errors(pred_w2c: np.ndarray, gt_w2c: np.ndarray) -> np.ndarray:
    """Angle (deg) between predicted and true relative rotations, all pairs i < j."""
    errors = []
    for i in range(len(pred_w2c)):
        for j in range(i + 1, len(pred_w2c)):
            rel_pred = pred_w2c[i, :, :3] @ pred_w2c[j, :, :3].T
            rel_gt = gt_w2c[i, :, :3] @ gt_w2c[j, :, :3].T
            errors.append(Rotation.from_matrix(rel_pred.T @ rel_gt).magnitude())
    return np.degrees(np.array(errors))


def depth_metrics(pred: np.ndarray, gt: np.ndarray) -> tuple[float, float]:
    """Median-scaled AbsRel and delta < 1.25 for one (H, W) depth map.

    Pixels with gt == 0 are ignored.
    """
    valid = gt > 0
    pred, gt = pred[valid], gt[valid]
    pred = pred * np.median(gt) / np.median(pred)
    abs_rel = np.mean(np.abs(pred - gt) / gt)
    delta1 = np.mean(np.maximum(pred / gt, gt / pred) < 1.25)
    return float(abs_rel), float(delta1)


def chamfer_distance(
    cloud_a: torch.Tensor, cloud_b: torch.Tensor, chunk: int = 4096
) -> float:
    """Symmetric mean nearest-neighbour distance between (n, 3) and (m, 3) clouds."""

    def one_way(src: torch.Tensor, dst: torch.Tensor) -> torch.Tensor:
        """Mean distance from each src point to its nearest dst point."""
        dists = [
            torch.cdist(src[i : i + chunk], dst).min(dim=1).values
            for i in range(0, len(src), chunk)
        ]
        return torch.cat(dists).mean()

    return float(0.5 * (one_way(cloud_a, cloud_b) + one_way(cloud_b, cloud_a)))

Sanity checks on ground truth: perfect predictions score zero, and a known 1 cm shift gives just
under 1 cm Chamfer distance.

In [ ]:
gt_cloud = gt_points[gt_valid][::40].float()
shifted = gt_cloud + torch.tensor([0.01, 0.0, 0.0])
print(
    "ATE(gt, gt)            = "
    f"{absolute_trajectory_error(gt_centres, gt_centres):.2e} m"
)
print(
    f"max RRE(gt, gt)        = {relative_rotation_errors(gt_w2c, gt_w2c).max():.2e} deg"
)
print(
    "depth(2 x gt, gt)      = AbsRel "
    f"{depth_metrics(2 * gt_depth[0], gt_depth[0])[0]:.2e}"
)
print(
    f"Chamfer(gt, gt + 1 cm) = {chamfer_distance(gt_cloud, shifted) * 100:.2f} cm "
    f"on {len(gt_cloud)} points"
)

## 4.5 The confidence-weighted loss

The per-pixel objective from §3.4, without the gradient term. Sweeping $c$ for one fixed error
confirms the minimiser $c^\star = \alpha / e$, which is the property §5.5 tests on real outputs.

In [ ]:
def confidence_loss(
    pred: torch.Tensor, target: torch.Tensor, conf: torch.Tensor, alpha: float = 0.2
) -> torch.Tensor:
    """pred, target (..., 3), conf (...) -> scalar.

    Mean of c * ||pred - target|| - alpha * log c.
    """
    error = (pred - target).norm(dim=-1)
    return (conf * error - alpha * conf.log()).mean()


alpha = 0.2
for error in [0.05, 0.2, 0.8]:
    pred = torch.tensor([[error, 0.0, 0.0]])
    target = torch.zeros(1, 3)
    grid = torch.linspace(0.05, 10.0, 4000)
    losses = torch.stack([confidence_loss(pred, target, c[None], alpha) for c in grid])
    print(
        f"error {error:4.2f}: argmin c = {grid[losses.argmin()]:.2f}, "
        f"alpha / error = {alpha / error:.2f}"
    )

---

# 5) End-to-end: one forward pass on 8 views

## 5.1 Running the network piece by piece

Calling the aggregator and heads separately makes the token layout visible and keeps each head in
fp32. The aggregator returns the tokens of the 4 layers the DPT heads read (indices 4, 11, 17, 23),
each the concatenation of the frame-attention and global-attention outputs, hence width
$2 \times 1024$. All other layers are `None` to save memory.

In [ ]:
@torch.no_grad()
def run_vggt(images: torch.Tensor) -> dict:
    """images (num_views, 3, H, W) in [0, 1] -> predictions on CPU, time and memory."""
    images = images.to(DEVICE)[None]
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    start = time.time()
    tokens, patch_start_idx = model.aggregator(images.half())
    tokens = [t.float() if t is not None else None for t in tokens]
    pose_enc = model.camera_head(tokens)[-1]
    depth, depth_conf = model.depth_head(
        tokens, images=images, patch_start_idx=patch_start_idx
    )
    points, points_conf = model.point_head(
        tokens, images=images, patch_start_idx=patch_start_idx
    )
    if DEVICE.type == "mps":
        torch.mps.synchronize()
    return {
        "tokens": [tuple(t.shape) if t is not None else None for t in tokens],
        "patch_start_idx": patch_start_idx,
        "pose_enc": pose_enc[0].cpu(),
        "depth": depth[0, ..., 0].cpu(),
        "depth_conf": depth_conf[0].cpu(),
        "points": points[0].cpu(),
        "points_conf": points_conf[0].cpu(),
        "seconds": time.time() - start,
        "memory_gb": device_memory_gb(),
    }


pred = run_vggt(images)
num_patches = (height // 14) * (width // 14)
print(f"tokens per frame: {pred['patch_start_idx']} special + {num_patches} patches")
print(f"cached layers   : {[i for i, t in enumerate(pred['tokens']) if t is not None]}")
print(
    f"layer-23 tokens : {pred['tokens'][23]}  (batch, "
    "num_views, tokens_per_frame, 2 * embed_dim)"
)
print(
    f"pose_enc {tuple(pred['pose_enc'].shape)} | depth {tuple(pred['depth'].shape)} | "
    f"points {tuple(pred['points'].shape)}"
)
print(
    f"forward pass {pred['seconds']:.1f} s | device memory {pred['memory_gb']:.2f} GB"
)

## 5.2 Cameras

Decoded with the from-scratch decoder from §4.1, then scored against motion capture.

In [ ]:
pred_w2c, pred_K = decode_camera(pred["pose_enc"].double(), height, width)
pred_w2c, pred_K = pred_w2c.numpy(), pred_K.numpy()
pred_centres = camera_centres(pred_w2c)

ate = absolute_trajectory_error(pred_centres, gt_centres)
rre = relative_rotation_errors(pred_w2c, gt_w2c)
path_length = np.linalg.norm(np.diff(gt_centres, axis=0), axis=1).sum()
print(f"ATE        : {ate * 100:.2f} cm over a {path_length:.2f} m path")
print(
    f"RRE        : mean {rre.mean():.2f} deg, max "
    f"{rre.max():.2f} deg over {len(rre)} pairs"
)
print(
    f"focal f_x  : predicted {pred_K[:, 0, 0].mean():.1f} "
    f"± {pred_K[:, 0, 0].std():.1f} px, "
    f"calibrated {gt_K[0, 0]:.1f} px"
)

In [ ]:
scale, rot, trans = umeyama(pred_centres, gt_centres)
aligned_centres = scale * pred_centres @ rot.T + trans

fig, ax = plt.subplots(figsize=(6, 5))
# top-down view: x (right) against z (forward) of the first camera
ax.plot(gt_centres[:, 0], gt_centres[:, 2], "o-", label="motion capture")
ax.plot(
    aligned_centres[:, 0], aligned_centres[:, 2], "x--", label="VGGT (Umeyama-aligned)"
)
for i, (x, z) in enumerate(gt_centres[:, [0, 2]]):
    ax.annotate(str(i), (x, z), textcoords="offset points", xytext=(4, 4))
ax.set_xlabel("x (m)")
ax.set_ylabel("z (m)")
ax.set_title("Camera centres in the first camera's frame")
ax.axis("equal")
ax.legend()
plt.show()

## 5.3 Depth

Each predicted map is median-scaled to the Kinect map (§3.6), since depth is up to scale.

In [ ]:
pred_depth = pred["depth"].numpy()
scores = np.array([depth_metrics(pred_depth[i], gt_depth[i]) for i in range(NUM_VIEWS)])
print(
    f"AbsRel {scores[:, 0].mean():.3f} | delta1 "
    f"{scores[:, 1].mean():.3f}  (mean over {NUM_VIEWS} views)"
)

view = 3
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
panels = [
    (images[view].permute(1, 2, 0), "image", None),
    (
        np.where(gt_depth[view] > 0, gt_depth[view], np.nan),
        "Kinect depth (m)",
        "viridis",
    ),
    (pred_depth[view], "VGGT depth (relative)", "viridis"),
    (np.log(pred["depth_conf"][view].numpy()), "log confidence", "magma"),
]
for ax, (data, title, cmap) in zip(axes, panels):
    shown = ax.imshow(data, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    if cmap:
        fig.colorbar(shown, ax=ax, fraction=0.035)
plt.tight_layout()
plt.show()

## 5.4 Point maps: point head vs depth + camera

Both clouds are aligned to the ground-truth cloud by one similarity transform, fit with Umeyama on
pixel correspondences (the same pixel gives a predicted and a true 3D point). Error is then the
per-pixel distance in metres.

In [ ]:
def aligned_point_error(
    pred_points: np.ndarray, gt_points: np.ndarray, valid: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """Similarity-align (num_views, H, W, 3) predictions to gt on valid pixels.

    Returns per-point errors and the aligned cloud.
    """
    src, dst = pred_points[valid], gt_points[valid]
    sample = np.random.default_rng(SEED).choice(
        len(src), size=min(50_000, len(src)), replace=False
    )
    scale, rot, trans = umeyama(src[sample], dst[sample])
    aligned = scale * src @ rot.T + trans
    return np.linalg.norm(aligned - dst, axis=1), aligned


depth_route = unproject_depth(
    pred["depth"].double(), torch.from_numpy(pred_w2c), torch.from_numpy(pred_K)
).numpy()
point_route = pred["points"].double().numpy()
gt_np = gt_points.numpy()

for name, cloud in [("point head", point_route), ("depth + camera", depth_route)]:
    errors, aligned = aligned_point_error(cloud, gt_np, gt_valid)
    chamfer = chamfer_distance(torch.from_numpy(aligned[::40]).float(), gt_cloud)
    print(
        f"{name:15s} median error {np.median(errors) * 100:5.2f} cm | "
        f"mean {errors.mean() * 100:5.2f} cm | Chamfer {chamfer * 100:5.2f} cm"
    )

In [ ]:
_, depth_aligned = aligned_point_error(depth_route, gt_np, gt_valid)
colours = images.permute(0, 2, 3, 1).numpy()[gt_valid]
keep = np.random.default_rng(SEED).choice(
    len(depth_aligned), size=20_000, replace=False
)

fig = plt.figure(figsize=(12, 5))
for k, (cloud, title) in enumerate(
    [(gt_np[gt_valid], "Kinect + mocap"), (depth_aligned, "VGGT, aligned")]
):
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    ax.scatter(cloud[keep, 0], cloud[keep, 2], -cloud[keep, 1], c=colours[keep], s=0.5)
    ax.set_title(title)
    ax.view_init(elev=25, azim=-70)
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## 5.5 Does confidence predict error?

§3.4 says a well-trained model outputs $c_p \approx \alpha / e_p$. Sorting pixels by predicted
depth confidence and measuring the real depth error in each fifth checks that directly.

In [ ]:
depth_conf = pred["depth_conf"].numpy()[gt_valid]
pred_scaled = (
    pred_depth[gt_valid]
    * np.median(gt_depth[gt_valid])
    / np.median(pred_depth[gt_valid])
)
rel_error = np.abs(pred_scaled - gt_depth[gt_valid]) / gt_depth[gt_valid]

edges = np.quantile(depth_conf, np.linspace(0, 1, 6))
print("confidence quintile | mean conf | depth AbsRel")
for q in range(5):
    in_bin = (depth_conf >= edges[q]) & (depth_conf <= edges[q + 1])
    print(
        f"        Q{q + 1}          | {depth_conf[in_bin].mean():9.2f} "
        f"| {rel_error[in_bin].mean():.3f}"
    )
rank_corr = np.corrcoef(
    np.argsort(np.argsort(depth_conf)), np.argsort(np.argsort(rel_error))
)[0, 1]
print(f"Spearman(conf, error) = {rank_corr:.3f}")

---

# 6) Variants

## 6.1 Classical two-view baseline

The classical way to get a relative pose is ORB features, brute-force matching, then a
RANSAC-fitted essential matrix decomposed into $R$ and $t$. It is given the **calibrated**
intrinsics, an advantage VGGT does not have. `fr1/desk` is a fast hand-held sweep with heavy motion
blur, which is hard on keypoints. Both methods are scored with the relative rotation
error on every pair among the 8 views. Pairs further apart have wider baselines and less overlap.

In [ ]:
def classical_relative_rotation(path_i: Path, path_j: Path) -> np.ndarray | None:
    """ORB + RANSAC essential matrix on raw 640 x 480 frames -> R_ij (3, 3) or None."""
    gray_i = cv2.imread(str(path_i), cv2.IMREAD_GRAYSCALE)
    gray_j = cv2.imread(str(path_j), cv2.IMREAD_GRAYSCALE)
    orb = cv2.ORB_create(nfeatures=4000)
    kp_i, desc_i = orb.detectAndCompute(gray_i, None)
    kp_j, desc_j = orb.detectAndCompute(gray_j, None)
    matches = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True).match(desc_i, desc_j)
    if len(matches) < 8:
        return None
    pts_i = np.float64([kp_i[m.queryIdx].pt for m in matches])
    pts_j = np.float64([kp_j[m.trainIdx].pt for m in matches])
    E, inliers = cv2.findEssentialMat(
        pts_i, pts_j, TUM_K, method=cv2.RANSAC, threshold=1.0
    )
    if E is None or E.shape != (3, 3):
        return None
    _, rot, _, _ = cv2.recoverPose(E, pts_i, pts_j, TUM_K, mask=inliers)
    # recoverPose maps camera i to camera j: x_j = R x_i + t, i.e. R_j R_i^T
    return rot


rows = []
for i in range(NUM_VIEWS):
    for j in range(i + 1, NUM_VIEWS):
        rel_gt = gt_w2c[j, :, :3] @ gt_w2c[i, :, :3].T
        rel_vggt = pred_w2c[j, :, :3] @ pred_w2c[i, :, :3].T
        rel_orb = classical_relative_rotation(views[i]["rgb"], views[j]["rgb"])
        err_vggt = np.degrees(Rotation.from_matrix(rel_vggt.T @ rel_gt).magnitude())
        err_orb = (
            np.inf
            if rel_orb is None
            else np.degrees(Rotation.from_matrix(rel_orb.T @ rel_gt).magnitude())
        )
        rows.append((j - i, err_orb, err_vggt))
rows = np.array(rows)

print("frame gap | pairs | ORB+RANSAC median RRE | VGGT median RRE")
for gap in [1, 2, 4, 7]:
    sel = rows[:, 0] == gap
    print(
        f"{gap:9d} | {sel.sum():5d} | {np.median(rows[sel, 1]):17.2f} "
        f"deg | {np.median(rows[sel, 2]):9.2f} deg"
    )
print(
    f"all pairs: ORB {np.median(rows[:, 1]):.2f} "
    f"deg, VGGT {np.median(rows[:, 2]):.2f} deg; "
    f"ORB > 5 deg on {np.mean(rows[:, 1] > 5):.0%} "
    f"of pairs, VGGT on {np.mean(rows[:, 2] > 5):.0%}"
)

## 6.2 Accuracy, time and memory vs number of views

Global attention costs $O\big((S \cdot N)^2\big)$ for $S$ frames of $N$ tokens, so time should grow
faster than linearly in $S$. More views over the **same** stretch of trajectory give the network
more constraints. ATE is omitted for 2 views because a similarity transform fits two centres
exactly.

In [ ]:
print("views | time (s) | memory (GB) | ATE (cm) | mean RRE (deg) | AbsRel")
for num_views in [2, 4, 8, 12]:
    idx = np.linspace(0, FRAME_SPAN, num_views).round().astype(int)
    sweep_views = [frames[i] for i in idx]
    sweep_images = load_and_preprocess_images([str(v["rgb"]) for v in sweep_views])
    out = run_vggt(sweep_images)
    w2c, _ = decode_camera(out["pose_enc"].double(), height, width)
    w2c = w2c.numpy()
    c2w_gt = np.stack([pose_to_c2w(v["pose"]) for v in sweep_views])
    w2c_gt = np.linalg.inv(np.linalg.inv(c2w_gt[0]) @ c2w_gt)[:, :3]
    depth_gt = np.stack([load_gt_depth(v["depth"], height, width) for v in sweep_views])
    ate_cm = (
        absolute_trajectory_error(camera_centres(w2c), camera_centres(w2c_gt)) * 100
        if num_views > 2
        else float("nan")
    )
    abs_rel = np.mean(
        [
            depth_metrics(out["depth"][k].numpy(), depth_gt[k])[0]
            for k in range(num_views)
        ]
    )
    print(
        f"{num_views:5d} | {out['seconds']:8.1f} | "
        f"{out['memory_gb']:11.2f} | {ate_cm:8.2f} | "
        f"{relative_rotation_errors(w2c, w2c_gt).mean():14.2f} | {abs_rel:.3f}"
    )

---

# 7) Practical notes

**Memory and frame count.** Global attention over $S \times 1041$ tokens dominates, and the two DPT
heads add full-resolution activations per frame. On a single 80 GB GPU the 1B model handles a few
hundred frames. On a 16 GB laptop, a dozen at 518 px is a comfortable ceiling. The DPT heads
process frames in chunks (`frames_chunk_size`) to cap their activations, and dropping the heads you
do not need (`enable_point=False`, `enable_track=False`) is the cheapest saving.

**Scale and gauge.** Outputs are in the first camera's frame and an arbitrary global scale. Anything
metric (robot navigation, AR placement) needs one external measurement: a known object size, an
IMU, or a metric depth sensor. Choosing a different first frame changes the world frame, so
compare cameras with relative metrics (RRE) or after alignment (ATE).

**Which output to trust.** Use depth + camera for the cloud, and filter by confidence. §5.5 shows the
confidence ranking is informative, so dropping the lowest-confidence 20–30 % of pixels removes
much of the error at the cost of coverage.

**Memory here.** One pass over 8 views peaked at about 8 GB on a 16 GB Mac (§5.1), and 12 views at
about 8.5 GB. Close other GPU work before running it.

**Failure modes.** Texture-less or reflective surfaces (monitors, glass, glossy desks), dynamic
objects (the network assumes a static scene), and extreme intrinsics (fisheye) all degrade
predictions. Confidence drops in those regions, which is the cue to mask them. TUM's Kinect frames
are not undistorted, which slightly inflates the error measured here.

**Hand-off to Gaussian splatting.** Production pipelines (VGGS, feed-forward 3DGS) use VGGT's
cameras and confidence-filtered points to **initialise** a 3D Gaussian splatting optimisation
instead of COLMAP. That removes the SfM step and starts from dense, well-placed Gaussians, so the
optimisation converges several times faster. Rasterising Gaussians needs a CUDA kernel (`gsplat`),
so it is out of scope on this machine. The input it would take is exactly `depth_aligned`,
`pred_w2c` and `pred_K` above.

**Refinement.** VGGT's predictions are already close, but bundle adjustment on top (using its own
track head for correspondences) still reduces pose error further. The paper uses this for its best
benchmark numbers at the cost of seconds of optimisation.

---

# 8) Conclusion

| Question | Classical SfM + MVS | Feed-forward (VGGT) |
|---|---|---|
| Inputs needed | Many views with good overlap; often intrinsics | 1 to hundreds of unposed images |
| Time per scene | Minutes to hours | One forward pass (seconds) |
| Works with 2–5 views | Often fails to initialise | Yes; accuracy improves with more views |
| Metric scale | No (without extra sensors) | No: up to scale, in camera-1 frame |
| Per-pixel uncertainty | Reprojection residuals, sparse | Dense confidence that ranks error |
| Best use | Large, well-captured scenes needing maximum accuracy | Fast sparse-view reconstruction, initialising 3DGS or bundle adjustment |